In [ ]:
%%configure -f
{
    "driverMemory": "4G",
    "executorCores": 12,
    "executorMemory": "12G",
    "numExecutors": 3,
    "conf": {
        "spark.dynamicAllocation.enabled": "false",
        "spark.jars.packages": "org.apache.hudi:hudi-spark-bundle_2.12:0.14.1",
        "spark.pyspark.python": "python3",
        "spark.pyspark.virtualenv.bin.path": "/usr/bin/virtualenv",
        "spark.pyspark.virtualenv.enabled": "true",
        "spark.pyspark.virtualenv.type": "native",
        "spark.serializer": "org.apache.spark.serializer.KryoSerializer",
        "spark.sql.adaptive.coalescePartitions.enabled": "true",
        "spark.sql.adaptive.enabled": "true",
        "spark.sql.parquet.mergeSchema": "true",
        "spark.submit.pyFiles": "s3://destino/*.toml"
    }
}

In [ ]:
import sys
import time
import datetime
from pyspark.sql.functions import *
from pyspark.sql.functions import sum as spark_sum
from pyspark.sql.session import *
from pyspark.sql.types import *
from pyspark.sql.window import *
from agi_tools.spark import AgiTools

In [ ]:
agi = AgiTools(private=False)
spark = agi.create_spark_session('Refined_Interacoes_Nuvidio_Historico')

# Tabelas Raw Full

In [ ]:
#Chamadas
Refined_Interacoes_Nuvidio = spark.sql ("""
WITH 

CTE_CASOS as (
    select
        data_hora_de_abertura,
        numero_do_caso,
        origem_do_caso,
        motivo,
        tipo,
        produto,
        detalhe,
        cpf_cnpj
    from datalake_corbanmis_public.cb_salesforce_casos
    where motivo in('Chamada de Vídeo - Cíveis', 'Chamada de vídeo - Cíveis')
)

,CTE_SESSAO as(
    select 
        protocolo as cod_chamada,
        protocolo as cod_interacao_chamada,
        to_timestamp(datadeinicio) as dat_chamada_original, 
        cast(to_timestamp(datadeinicio) as date) as dat_chamada,
        DATE_FORMAT(DATE_TRUNC('HOUR', to_timestamp(clienteentrounafila)), 'HH:mm:ss') AS tmp_chamada,
        DATE_FORMAT(DATE_TRUNC('HOUR', to_timestamp(clienteentrounafila)) + (CASE WHEN minute(to_timestamp(clienteentrounafila)) < 30 THEN interval '0' minute ELSE interval '30' minute END), 'HH:mm:ss') AS tmp_chamada_30,
        'video' as dsc_tipo_midia,
        'inbound' as dsc_direcao,
        contatodocliente as dsc_telefone_cliente,
        fila as cod_fila,
        CASE 
            WHEN fila = 'chamada-video' THEN 'Fraude'
            WHEN fila = 'chamada-video-causa-civel' THEN 'Fraude Causa Cível'
            WHEN fila = 'chamada-video-celetista' THEN 'Fraude CLT'
            WHEN fila IS NULL THEN NULL
            ELSE 'Outros'
        END AS dsc_fila,
        case
            when emaildoatendente like 'aec%' then 'AEC' 
            when emaildoatendente like 'tel%' then 'TEL'
            when emaildoatendente like 'almaviva%' then 'ALMAVIVA'
            else 'AGIBANK'
        end as dsc_empresa,
        60 as num_ns,
        emaildoatendente as cod_agente,
        CASE 
            WHEN emaildoatendente LIKE '%@%' THEN emaildoatendente
            ELSE concat(emaildoatendente, '@agi.com.br')
        END as dsc_email,
        lower(atendente) as dsc_nome_agente_minusculo,
        true as flg_oferecida,
        false as flg_abandonada,
        true as flg_atendida_agente,
        false as flg_transferencia,
        case when (unix_timestamp(to_timestamp(clientesaiudachamada)) - unix_timestamp(to_timestamp(clienteentrounachamada))) <= 30 then true else false end as flg_shortcall,
        case when (unix_timestamp(to_timestamp(atendenteentrounachamada)) - unix_timestamp(to_timestamp(clienteentrounafila))) <= 60 then true else false end as flg_nivel_servico,
        duracaoemsegundos as tmp_operacional,
        duracaoemsegundos as tmp_interacao_agente,
        (unix_timestamp(to_timestamp(atendenteentrounachamada)) - unix_timestamp(to_timestamp(clienteentrounafila))) as tmp_espera,
        if(avaliacao is null, null, 1) as qtd_pesquisa,
        avaliacao as dsc_nota,
        -- Nessa base não tem pesquisa de resolutividade, mas tem que normalizar para casar com as outras.
        '' as dsc_resolutividade,
        CASE 
            WHEN (unix_timestamp(to_timestamp(atendenteentrounachamada)) - unix_timestamp(to_timestamp(clienteentrounafila))) < 10 THEN '01. <10s'
            WHEN (unix_timestamp(to_timestamp(atendenteentrounachamada)) - unix_timestamp(to_timestamp(clienteentrounafila))) < 20 THEN '02. 10s-20s'
            WHEN (unix_timestamp(to_timestamp(atendenteentrounachamada)) - unix_timestamp(to_timestamp(clienteentrounafila))) < 30 THEN '03. 20s-30s'
            WHEN (unix_timestamp(to_timestamp(atendenteentrounachamada)) - unix_timestamp(to_timestamp(clienteentrounafila))) < 40 THEN '04. 30s-40s'
            WHEN (unix_timestamp(to_timestamp(atendenteentrounachamada)) - unix_timestamp(to_timestamp(clienteentrounafila))) < 50 THEN '05. 40s-50s'
            WHEN (unix_timestamp(to_timestamp(atendenteentrounachamada)) - unix_timestamp(to_timestamp(clienteentrounafila))) < 60 THEN '06. 50s-1min'
            WHEN (unix_timestamp(to_timestamp(atendenteentrounachamada)) - unix_timestamp(to_timestamp(clienteentrounafila))) < 300 THEN '07. 1min-5min'
            WHEN (unix_timestamp(to_timestamp(atendenteentrounachamada)) - unix_timestamp(to_timestamp(clienteentrounafila))) < 600 THEN '08. 5min-10min'
            WHEN (unix_timestamp(to_timestamp(atendenteentrounachamada)) - unix_timestamp(to_timestamp(clienteentrounafila))) < 900 THEN '09. 10min-15min'
            WHEN (unix_timestamp(to_timestamp(atendenteentrounachamada)) - unix_timestamp(to_timestamp(clienteentrounafila))) < 1200 THEN '10. 15min-20min'
            ELSE '11. >20min'
        END AS dsc_faixa_espera,
        b.numero_do_caso as dsc_numero_caso,
        b.origem_do_caso as origem_do_caso,
        b.tipo as tipo,
        b.produto as produto,
        b.motivo as motivo,
        b.detalhe as detalhe,
        cpfdocliente as cpf_cnpj
    from datalake_nuvidio.chamadas_atendidas a
    -- Agregando dados de informação do caso atraves do numero do caso. # Fazendo o join nas duas tabelas, ao invés de fazer no final, mas considerando atualizar no futuro para tornar o processo mais performático, e trazer apenas os casos abertos nos últimos 6 meses, ao invés de puxar a tabela toda.
    left join (
        select 
            cpf_cnpj,
            data_hora_de_abertura,
            numero_do_caso,
            origem_do_caso,
            motivo,
            tipo,
            produto,
            detalhe,
            row_number() over(partition by cpf_cnpj order by data_hora_de_abertura desc) as rn
        from CTE_CASOS
    ) b on a.cpfdocliente = b.cpf_cnpj
        and to_timestamp(a.datadeinicio) >= b.data_hora_de_abertura
        and b.rn = 1

    union all

    select 
        ticket as cod_chamada,
        ticket as cod_interacao_chamada,
        to_timestamp(datadeentrada) as dat_chamada_original, 
        cast(to_timestamp(datadeentrada) as date) as dat_chamada,
        -- Intervalo de 1 hora
        DATE_FORMAT(DATE_TRUNC('HOUR', to_timestamp(clienteentrounafila)), 'HH:mm:ss') AS tmp_chamada,
        -- Intervalo de 30 em 30 min
        DATE_FORMAT(DATE_TRUNC('HOUR', to_timestamp(clienteentrounafila)) + (CASE WHEN minute(to_timestamp(clienteentrounafila)) < 30 THEN interval '0' minute ELSE interval '30' minute END), 'HH:mm:ss') AS tmp_chamada_30,
        'video' as dsc_tipo_midia,
        'inbound' as dsc_direcao,
        contatodocliente as dsc_telefone_cliente,
        null as cod_fila,
        null as dsc_fila,
        null as dsc_empresa, 
        60 as num_ns,
        null as cod_agente,
        null as dsc_email,
        null as dsc_nome_agente_minusculo,
        true as flg_oferecida,
        true as flg_abandonada,
        false as flg_atendida_agente,
        false as flg_transferencia,
        false as flg_shortcall,
        if(cast(esperaemsegundos as int) <= 60, true, false) as flg_nivel_servico,
        NULL as tmp_operacional, 
        NULL as tmp_interacao_agente,  
        cast(esperaemsegundos as bigint) as tmp_espera,
        null as qtd_pesquisa,
        null as dsc_nota,
        null as dsc_resolutividade,
        -- Faixa de Espera
        CASE 
            WHEN cast(esperaemsegundos as int) < 10 THEN '01. <10s'
            WHEN cast(esperaemsegundos as int) < 20 THEN '02. 10s-20s'
            WHEN cast(esperaemsegundos as int) < 30 THEN '03. 20s-30s'
            WHEN cast(esperaemsegundos as int) < 40 THEN '04. 30s-40s'
            WHEN cast(esperaemsegundos as int) < 50 THEN '05. 40s-50s'
            WHEN cast(esperaemsegundos as int) < 60 THEN '06. 50s-1min'
            WHEN cast(esperaemsegundos as int) < 300 THEN '07. 1min-5min'
            WHEN cast(esperaemsegundos as int) < 600 THEN '08. 5min-10min'
            WHEN cast(esperaemsegundos as int) < 900 THEN '09. 10min-15min'
            WHEN cast(esperaemsegundos as int) < 1200 THEN '10. 15min-20min'
            ELSE '11. >20min'
        END AS dsc_faixa_espera,
        -- Numero do caso no salesforce
        b.numero_do_caso as dsc_numero_caso,
        b.origem_do_caso as origem_do_caso,
        b.tipo as tipo,
        b.produto as produto,
        b.motivo as motivo,
        b.detalhe as detalhe,
        cpfdocliente as cpf_cnpj
    from datalake_nuvidio.chamadas_perdidas a
    -- Agregando dados de informação do caso atraves do numero do caso
    left join (
        select 
            cpf_cnpj,
            data_hora_de_abertura,
            numero_do_caso,
            origem_do_caso, 
            motivo,
            tipo,
            produto,
            detalhe,
            row_number() over(partition by cpf_cnpj order by data_hora_de_abertura desc) as rn
        from CTE_CASOS
    ) b on a.cpfdocliente = b.cpf_cnpj
        and to_timestamp(a.datadeentrada) >= b.data_hora_de_abertura
        and b.rn = 1
)

SELECT
    cod_chamada,
    cod_interacao_chamada,
    dat_chamada_original,
    dat_chamada,
    tmp_chamada,
    tmp_chamada_30,
    dsc_tipo_midia,
    dsc_direcao,
    dsc_telefone_cliente,
    cod_fila,
    dsc_fila,
    'Fraude' as dsc_celula,
    dsc_empresa,
    dsc_empresa as dsc_empresa_agente,
    num_ns,
    cod_agente,
    dsc_email,
    dsc_nome_agente_minusculo,
    flg_oferecida,
    flg_abandonada,
    flg_atendida_agente,
    flg_transferencia,
    flg_shortcall,
    flg_nivel_servico,
    -- Calculo de rechamada por numero de telefone. Rechamada no mesmo dia. Marca a rechamada na chamada que gerou a rechamada
    case when cast(dat_chamada_original as date) = lead(cast(dat_chamada_original as date)) over(partition by dsc_telefone_cliente order by dat_chamada_original) then true else false end as flg_rechamada_agente,
    cast(tmp_operacional as int),
    cast(tmp_interacao_agente as int),
    tmp_espera,
    IF(dsc_numero_caso IS NULL, NULL, 1) AS qtd_pesquisa,
    cast(dsc_nota as int),
    dsc_resolutividade,
    dsc_faixa_espera,
    dsc_numero_caso,
    origem_do_caso,
    tipo,
    produto,
    motivo,
    detalhe,
    cpf_cnpj
from cte_sessao
""")

In [ ]:
# Agrupar por dat_status e contar registros
result = Refined_Interacoes_Nuvidio.groupBy(col("dat_chamada")).agg(count("*").alias("count"))

# Ordenar pelo campo de data e exibir
result.orderBy("dat_chamada").show()

In [ ]:
#Parcial
spark.conf.set("spark.sql.sources.partitionOverwriteMode", "dynamic")
#Total
# spark.conf.set("spark.sql.sources.partitionOverwriteMode", "static")

In [ ]:
agi.insert_into_lake(
    data=Refined_Interacoes_Nuvidio
    #,table_name='corbanmis_teste' # vai salvar a tabela 'datalake_corbanmis_public.corbanmis_teste'
    #,write_mode='overwrite'       # opcional
    ,partition_by='dat_chamada' # opcional
)
agi.destroy_spark_session()